In [12]:
# Prac 4: Creating Policy table for grid problem using MDP
import numpy as np
import random

grid_size = 4
goal_size = (3,3)
goal_state = goal_size
actions = ['up', 'down', 'left', 'right']
action_map = {'up': (-1,0), 'down': (1,0), 'left': (0,-1), 'right': (0,1)}

#Parameters
theta = 1e-4 #Learning rate
gamma = 0.9  #Discount factor
rewards = np.zeros((grid_size,grid_size))
rewards[goal_state] = 10

#Helper function
def get_next_state(state, action):
    r, c = state
    dr, dc = action_map[action]
    new_r, new_c = r + dr, c + dc
    if 0 <= new_r < grid_size and 0 <= new_c < grid_size:
        return (new_r, new_c)
    return state # if out of bounds, stay in the same state

value_table = np.zeros((grid_size,grid_size))
value_table

while True:
    delta = 0
    new_value_table = np.copy(value_table)
    for r in range(grid_size):
        for c in range(grid_size):
            state = (r,c)
            if state == goal_state:
                continue # skip goal state

            # calc the value of the state using the bellmans eqn
            state_values = []
            for action in actions:
                next_state = get_next_state(state, action)
                reward = rewards[next_state]
                state_values.append(reward + gamma * value_table[next_state])

            # Update the value table
            new_value_table[state] = max(state_values)
            delta = max(delta, abs(new_value_table[state] - value_table[state]))

    value_table = new_value_table
    if delta < theta:
        break

print("optimal value table:")
print(value_table)


optimal value table:
[[ 5.9049  6.561   7.29    8.1   ]
 [ 6.561   7.29    8.1     9.    ]
 [ 7.29    8.1     9.     10.    ]
 [ 8.1     9.     10.      0.    ]]


In [13]:
# Prac 4: Creating Policy table for grid problem using MDP
import numpy as np
import random

grid_size = 4
goal_size = (3,3)
goal_state = goal_size
actions = ['up', 'down', 'left', 'right']
action_map = {'up': (-1,0), 'down': (1,0), 'left': (0,-1), 'right': (0,1)}

#Parameters
theta = 1e-4 #Learning rate
gamma = 0.9  #Discount factor
rewards = np.zeros((grid_size,grid_size))
rewards[goal_state] = 10

#Helper function
def get_next_state(state, action):
    r, c = state
    dr, dc = action_map[action]
    new_r, new_c = r + dr, c + dc
    if 0 <= new_r < grid_size and 0 <= new_c < grid_size:
        return (new_r, new_c)
    return state # if out of bounds, stay in the same state

value_table = np.zeros((grid_size,grid_size))
value_table

while True:
    delta = 0
    new_value_table = np.copy(value_table)
    for r in range(grid_size):
        for c in range(grid_size):
            state = (r,c)
            if state == goal_state:
                continue # skip goal state

            # calc the value of the state using the bellmans eqn
            state_values = []
            for action in actions:
                next_state = get_next_state(state, action)
                reward = rewards[next_state]
                state_values.append(reward + gamma * value_table[next_state])

            # Update the value table
            new_value_table[state] = max(state_values)
            delta = max(delta, abs(new_value_table[state] - value_table[state]))

    value_table = new_value_table
    if delta < theta:
        break

print("optimal value table:")
print(value_table)


optimal value table:
[[ 5.9049  6.561   7.29    8.1   ]
 [ 6.561   7.29    8.1     9.    ]
 [ 7.29    8.1     9.     10.    ]
 [ 8.1     9.     10.      0.    ]]


In [14]:
# Prac 5: Using Q-Learning and MDP on grid problem
import numpy as np
import random

# Environment setup
grid_size = 4
goal_state = (3, 3)
actions = ['up', 'down', 'left', 'right']
action_map = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}

# Parameters
alpha = 0.1      # learning rate
gamma = 0.9      # discount factor
epsilon = 0.1    # exploration rate
episodes = 1000  # number of training episodes

# Reward initialization
rewards = np.zeros((grid_size, grid_size))
rewards[goal_state] = 10


# Q-table initialization (3D: rows x cols x actions)
q_table = np.zeros((grid_size, grid_size, len(actions)))

# Value table initialization
value_table = np.zeros((grid_size, grid_size))

# Helper function to get next state
def get_next_state(state, action):
    row, col = state
    d_row, d_col = action_map[action]
    new_row = row + d_row
    new_col = col + d_col
    if 0 <= new_row < grid_size and 0 <= new_col < grid_size:
        return (new_row, new_col)
    return state  # if out of bounds, stay in the same state

# Q-learning algorithm
for episode in range(episodes):
    state = (0, 0)  # start state
    
    while state != goal_state:
        # Choose action (epsilon-greedy)
        if random.uniform(0, 1) < epsilon:
            action = random.choice(actions)  # explore
        else:
            action = actions[np.argmax(q_table[state[0], state[1]])]  # exploit
        
        # Take action and observe next state and reward
        next_state = get_next_state(state, action)
        reward = rewards[next_state]
        
        # Q-learning update
        action_index = actions.index(action)
        best_next_q = np.max(q_table[next_state[0], next_state[1]])
        q_table[state[0], state[1], action_index] += alpha * (
            reward + gamma * best_next_q - q_table[state[0], state[1], action_index]
        )
        
        # Update value table using Bellman equation
        state_values = []
        for a in actions:
            next_s = get_next_state(state, a)
            state_values.append(rewards[next_s] + gamma * np.max(q_table[next_s[0], next_s[1]]))
        value_table[state] = max(state_values)
        
        # Move to next state
        state = next_state

# Derive policy from Q-table
policy = np.zeros((grid_size, grid_size), dtype=str)
for r in range(grid_size):
    for c in range(grid_size):
        state = (r, c)
        if state == goal_state:
            policy[state] = 'G'  # Goal
            continue
        
        action_index = np.argmax(q_table[state[0], state[1]])
        policy[state] = actions[action_index][0].upper()  # First letter of action

# Print results
print("Q-learning Q-Table:")
print(q_table)

print("\nValue Table:")
print(value_table)

print('\nOptimal Policy from Q-Learning')
for row in policy:
    print(' '.join(row))

Q-learning Q-Table:
[[[ 5.14971224  3.93915053  4.94660659  5.9049    ]
  [ 5.64676281  3.96437807  5.03240448  6.561     ]
  [ 5.83969923  5.07160288  5.428462    7.29      ]
  [ 6.82005243  8.1         6.33674907  6.93311467]]

 [[ 5.12460967  0.08919182  0.44258262  0.23351193]
  [ 5.5426878   0.          1.00301316  0.15148857]
  [ 6.51840724  1.58883955  1.10652299  0.81      ]
  [ 6.90162667  9.          5.19675186  7.81945889]]

 [[ 1.06956585  0.          0.          0.        ]
  [ 0.48175483  0.          0.          0.        ]
  [ 0.32863222  0.          0.          8.6566316 ]
  [ 7.35084813 10.          6.57400988  8.61619189]]

 [[ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]]]

Value Table:
[[ 5.9049      6.561       7.29        8.1       ]
 [ 5.31441     5.9049      8.1         9.        ]
 [ 4.38614367  7.4538917

In [11]:
import requests
from bs4 import BeautifulSoup

url = 'https://hastebin.com/share/karolarujo.python'

response = requests.get(url)
data = BeautifulSoup(response.text, 'html.parser')

print(data.get_text())

HastebinHastebin1# Prac 1: Q-Learning with Linear grid
import numpy as np

#define the environment(gridworld)
n_states = 5 #Number of states (0,1,2,3,4)
actions = [0,1] # Actions: 0=left , 1=right
rewards = [-1,-1,-1,-1,10]
goal_state =4 #Goal state
alpha = 0.1 #Learning rate
gamma = 0.9 #Discount Factor
epsilon = 0.1 #Exploration probability

#Initialize Q-table (n_states x actions)
q_table = np.zeros((n_states, len(actions)))
q_table

n_episodes = 1000
for episode in range(n_episodes):
    state = 0 #Start state
    while state != goal_state:
        #choose action (epsilon-greedy)
        if np.random.rand() < epsilon:
            action = np.random.choice(actions)
        else:
            action = np.argmax(q_table[state])
        #Take action and observe new state and reward
        new_state = state + 1 if action == 1 else max(0, state-1)
        reward = rewards[new_state]
        #Update Q-value using the Q-Learning formula
        q_table[state, action] += alpha * (reward + g

In [15]:
# Prac 6: Demonstrating solving 3x3 Grid problem using SARSA Approach
import random

ROWS, COLS = 3, 3

ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]

Q = {}
for r in range(ROWS):
    for c in range(COLS):
        Q[(r, c)] = {a: 0.0 for a in ACTIONS}
print(Q)

alpha = 0.1    # learning rate
gamma = 0.9    # discount factor
epsilon = 0.2  # exploration rate
episodes = 100

def choose_action(state):
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    return max(Q[state], key=Q[state].get)

def step(state, action):
    r, c = state

    if action == "UP" and r > 0:
        r -= 1
    elif action == "DOWN" and r < ROWS - 1:
        r += 1
    elif action == "LEFT" and c > 0:
        c -= 1
    elif action == "RIGHT" and c < COLS - 1:
        c += 1

    next_state = (r, c)

    if next_state == (0, 2):
        return next_state, 10, True
    else:
        return next_state, -1, False

for ep in range(episodes):
    state = (0, 0)
    action = choose_action(state)
    done = False

    while not done:
        next_state, reward, done = step(state, action)

        if not done:
            next_action = choose_action(next_state)
            next_q = Q[next_state][next_action]
        else:
            next_action = None
            next_q = 0
            
        Q[state][action] += alpha * (reward + gamma * next_q - Q[state][action])
        state = next_state
        action = next_action

print("Learned Q-values:\n")
for state in Q:
    print(state, Q[state])

policy = {}
for state in Q:
    policy[state] = max(Q[state], key=Q[state].get)
print("\nPolicy Table (Best action per state):")
for state in sorted(policy):
    print(state, "->", policy[state])
    

{(0, 0): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (0, 1): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (0, 2): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (1, 0): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (1, 1): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (1, 2): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (2, 0): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (2, 1): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}, (2, 2): {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}}
Learned Q-values:

(0, 0) {'UP': 0.22644426004032148, 'DOWN': -0.6015561744866029, 'LEFT': 0.5506817460843507, 'RIGHT': 7.114255640924123}
(0, 1) {'UP': 2.7607314083705345, 'DOWN': -0.39620554541215514, 'LEFT': 1.2225537349970386, 'RIGHT': 9.999550180377524}
(0, 2) {'UP': 0.0, 'DOWN': 0.0, 'LEFT': 0.0, 'RIGHT': 0.0}
(1, 0) {'UP': 0.7849320639730637, 'DOWN': -0.36910000000000004, 'LEFT': -0.2962, 'RIGHT': -0.2983718399544939}
(1, 1) {'UP': 1.99373

In [16]:
# Prac 7: Demonstrate Monte Carlo on Grid
import random
from collections import defaultdict

num_simulations = 10000
grid_size = (3, 3)
Q = defaultdict(lambda: defaultdict(float))

returns = defaultdict(lambda: defaultdict(list))
actions = {"down": (1, 0), "right": (0, 1)}

actions

for _ in range(num_simulations):
    start_x = random.randint(0, grid_size[0] - 1)
    start_y = random.randint(0, grid_size[1] - 1)
    current_position = (start_x, start_y)
    while current_position == (2, 2):
        start_x = random.randint(0, grid_size[0] - 1)
        start_y = random.randint(0, grid_size[1] - 1)
        current_position = (start_x, start_y)
    episode = []
    path_length = 0
    while current_position != (2, 2):
        valid_actions = []
        for action, (dx, dy) in actions.items():
            new_x, new_y = current_position[0] + dx, current_position[1] + dy
            if 0 <= new_x < grid_size[0] and 0 <= new_y < grid_size[1]:
                valid_actions.append(action)
        chosen_action = random.choice(valid_actions)
        dx, dy = actions[chosen_action]

        next_position = (current_position[0] + dx, current_position[1] + dy)
        episode.append((current_position, chosen_action))
        current_position = next_position
        path_length += 1
    G = path_length
    for state, action in episode:
        if G not in returns[state][action]:
            returns[state][action].append(G)

        Q[state][action] = sum(returns[state][action]) / len(returns[state][action])

for state, actions in Q.items():
    for action, value in actions.items():
        print(f"state: {state}, Action: {action}: {value:.2f}")

state: (2, 0), Action: right: 3.00
state: (2, 1), Action: right: 2.50
state: (1, 0), Action: right: 3.50
state: (1, 0), Action: down: 3.50
state: (1, 1), Action: right: 3.00
state: (1, 1), Action: down: 3.00
state: (1, 2), Action: down: 2.50
state: (0, 2), Action: down: 3.00
state: (0, 1), Action: right: 3.50
state: (0, 1), Action: down: 3.50
state: (0, 0), Action: right: 4.00
state: (0, 0), Action: down: 4.00


In [1]:
import requests
from bs4 import BeautifulSoup

url = 'https://hastebin.com/share/karolarujo.python'

response = requests.get(url)

data = BeautifulSoup(response.text, 'html.parser')

print(data.get_text())

HastebinHastebin1# Prac 1: Q-Learning with Linear grid
import numpy as np

#define the environment(gridworld)
n_states = 5 #Number of states (0,1,2,3,4)
actions = [0,1] # Actions: 0=left , 1=right
rewards = [-1,-1,-1,-1,10]
goal_state =4 #Goal state
alpha = 0.1 #Learning rate
gamma = 0.9 #Discount Factor
epsilon = 0.1 #Exploration probability

#Initialize Q-table (n_states x actions)
q_table = np.zeros((n_states, len(actions)))
q_table

n_episodes = 1000
for episode in range(n_episodes):
    state = 0 #Start state
    while state != goal_state:
        #choose action (epsilon-greedy)
        if np.random.rand() < epsilon:
            action = np.random.choice(actions)
        else:
            action = np.argmax(q_table[state])
        #Take action and observe new state and reward
        new_state = state + 1 if action == 1 else max(0, state-1)
        reward = rewards[new_state]
        #Update Q-value using the Q-Learning formula
        q_table[state, action] += alpha * (reward + g

In [4]:
import numpy as np
import random

In [6]:
n_states = 5
actions = [0,1]
rewards = [-1, -1, -1, -1 ,10]
goal_state = 4
alpha = 0.1
gamma = 0.9
epsilon = 0.1

In [7]:
q_table = np.zeros((n_states, len(actions)))
q_table

array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]])

In [ ]:
n_episodes = 100
for episode in range(n_episodes):
    state = 0
    while state

In [8]:
#define the environment(gridworld)
n_states = 5 #Number of states (0,1,2,3,4)
actions = [0,1] # Actions: 0=left , 1=right
rewards = [-1,-1,-1,-1,10]
goal_state =4 #Goal state
alpha = 0.1 #Learning rate
gamma = 0.9 #Discount Factor
epsilon = 0.1 #Exploration probability

#Initialize Q-table (n_states x actions)
q_table = np.zeros((n_states, len(actions)))
q_table

n_episodes = 1000
for episode in range(n_episodes):
    state = 0 #Start state
    while state != goal_state:
        #choose action (epsilon-greedy)
        if np.random.rand() < epsilon:
            action = np.random.choice(actions)
        else:
            action = np.argmax(q_table[state])
        #Take action and observe new state and reward
        new_state = state + 1 if action == 1 else max(0, state-1)
        reward = rewards[new_state]
        #Update Q-value using the Q-Learning formula
        q_table[state, action] += alpha * (reward + gamma * np.max(q_table[new_state]) - q_table[state, action])
        state = new_state

print(q_table)

state = 0
path = [state]
while state != goal_state:
    action = np.argmax(q_table[state])
    state = state+1 if action==1 else max(0,state-1)
    path.append(state)

print("Policy is",path)

[[ 3.11462191  4.58      ]
 [ 3.09311669  6.2       ]
 [ 4.56856096  8.        ]
 [ 6.18961552 10.        ]
 [ 0.          0.        ]]
Policy is [0, 1, 2, 3, 4]


In [9]:
# Prac 2: Creating Q-table and setting optimal path using Q-learning on 4x4 grid
import numpy as np
import random

grid_size = 4
goal_size = (3,3)
goal_state = goal_size
actions = ['up', 'down', 'left', 'right']
action_map = {'up': (-1,0), 'down': (1,0), 'left': (0,-1), 'right': (0,1)}

#Parameters
alpha = 0.1 #Learning rate
gamma = 0.9  #Discount factor
epsilon = 0.1  #Exploration rate
episodes = 1000

#Initialize Q-table
q_table = np.zeros((grid_size,grid_size,len(actions)))
q_table

#Helper function
def get_next_state(state, action):
    r, c = state
    dr, dc = action_map[action]
    new_r, new_c = r + dr, c + dc
    if 0 <= new_r < grid_size and 0 <= new_c < grid_size:
        return (new_r, new_c)
    return state # if out of bounds, stay in the same state

def get_reward(state):
    return 10 if state == goal_state else 0

# Q-Learning Algo
for episode in range(episodes):
    state = (0,0)
    while state != goal_state:
        if random.uniform(0,1) < epsilon:
            action = random.choice(actions)  # explore
        else:
            action = actions[np.argmax(q_table[state[0],state[1]])] # exploit

        # take action
        next_state = get_next_state(state, action)
        reward = get_reward(next_state)

        # update Q-value
        action_index = actions.index(action)
        best_next_q = np.max(q_table[next_state[0],next_state[1]])
        q_table[state[0],state[1],action_index] += alpha * (reward + gamma * best_next_q - q_table[state[0],state[1],action_index])

        state = next_state

# print Q-table
print("Trained Q-table:")
print(q_table)

# Test the agent
state = (0,0)
path = [state]
while state != goal_state:
    action = actions[np.argmax(q_table[state[0],state[1]])]
    state = get_next_state(state, action)
    path.append(state)

print("Optimal path:")
print(path)

Trained Q-table:
[[[ 4.81033324  4.14449656  4.94854752  5.9049    ]
  [ 5.20052654  3.81126977  4.9061596   6.561     ]
  [ 6.34541531  7.12756245  5.38811669  7.29      ]
  [ 6.95270995  8.1         5.67853418  6.67085465]]

 [[ 5.17974053  0.          0.75533772  0.        ]
  [ 5.52826762  0.          0.46167104  0.        ]
  [ 1.0266926   0.56131802  1.64063142  8.09969938]
  [ 7.10153811  9.          6.95193761  7.87120532]]

 [[ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]
  [ 6.85046871  0.          0.          0.9       ]
  [ 7.77630135 10.          5.04416735  8.54097758]]

 [[ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]
  [ 0.          0.          0.          0.        ]]]
Optimal path:
[(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (3, 3)]


In [10]:
# Prac 3: Applying MDP on Coin Game
import numpy as np

states = ['Start', 'Coin', 'Enemy']
actions = ['Go_Coin', 'Go_Enemy']

transition_rewards = {
    'Start': {'Go_Coin': 10, 'Go_Enemy': -5},
    'Coin': {},
    'Enemy': {}
}

transition_probs = {
    'Start': {'Go_Coin': {'Coin': 1.0}, 'Go_Enemy': {'Enemy': 1.0}},
    'Coin': {},
    'Enemy': {}
}

states, actions

transition_rewards

transition_probs

gamma = 0.9
theta = 0.001

value_function = {state: 0 for state in states}
value_function

while True:
    delta = 0
    new_value_function = value_function.copy()
    for state in states:
        if state in ['Coin', 'Enemy']:
            continue
        state_values = []
        for action in actions:
            value = 0
            for next_state, prob in transition_probs[state][action].items():
                reward = transition_rewards[state][action]
                print(reward)
                print(prob)
                print(value_function[next_state])
                value += prob * (reward + gamma * value_function[next_state])
                print(value)

            state_values.append(value)
            print("s_v", state_values)
        new_value_function[state] = max(state_values)
        print("n_v", new_value_function)
        delta = max(delta, abs(new_value_function[state] - value_function[state]))

    value_function = new_value_function
    print("v_f", value_function)
    if delta < theta:
        break

value_function

policy = {}
for state in states:
    if state in ['Coin', 'Enemy']:
        policy[state] = 'T'
        continue
    action_values = {}
    for action in actions:
        value = 0
        for next_state, prob in transition_probs[state][action].items():
            reward = transition_rewards[state][action]
            value += prob * (reward + gamma * value_function[next_state])
        action_values[action] = value
    policy[state] = max(action_values, key=action_values.get)
print("Optimal value function:", value_function)
print("Optimal policy:", policy)

10
1.0
0
10.0
s_v [10.0]
-5
1.0
0
-5.0
s_v [10.0, -5.0]
n_v {'Start': 10.0, 'Coin': 0, 'Enemy': 0}
v_f {'Start': 10.0, 'Coin': 0, 'Enemy': 0}
10
1.0
0
10.0
s_v [10.0]
-5
1.0
0
-5.0
s_v [10.0, -5.0]
n_v {'Start': 10.0, 'Coin': 0, 'Enemy': 0}
v_f {'Start': 10.0, 'Coin': 0, 'Enemy': 0}
Optimal value function: {'Start': 10.0, 'Coin': 0, 'Enemy': 0}
Optimal policy: {'Start': 'Go_Coin', 'Coin': 'T', 'Enemy': 'T'}
